# Weight Initialization & Batch Normalization Mechanics

This notebook demonstrates the critical importance of proper weight initialization (Xavier and He) in preventing exploding and vanishing activations across deep neural networks, and showcases the training stability provided by **Batch Normalization**.

## 1. Import Dependencies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

## 2. Forward Signal Variance Propagation (NumPy Simulation)

Let's pass input data through a 10-layer network containing 500 units per layer, and record the variance of activations at each layer using different initialization schemes.
- **Zero Initialization:** Sets all weights to 0. Activations become exactly 0, killing signal propagation.
- **Unscaled Normal:** $W \sim \mathcal{N}(0, 1)$. Activations explode exponentially.
- **Xavier (Glorot):** $W \sim \mathcal{N}(0, \frac{1}{n_{\text{in}}})$. Maintains variance for linear/tanh networks.
- **He (Kaiming):** $W \sim \mathcal{N}(0, \frac{2}{n_{\text{in}}})$. Maintains variance for ReLU networks.

In [ ]:
np.random.seed(42)
m = 1000
n_features = 500

# Input data: variance = 1.0
X = np.random.randn(n_features, m)

def simulate_propagation(init_mode, activation_fn):
    A = X
    variances = [np.var(A)]
    
    for l in range(10):
        n_in = A.shape[0]
        n_out = n_features
        
        if init_mode == 'zero':
            W = np.zeros((n_out, n_in))
        elif init_mode == 'unscaled':
            W = np.random.randn(n_out, n_in)
        elif init_mode == 'xavier':
            W = np.random.randn(n_out, n_in) * np.sqrt(1.0 / n_in)
        elif init_mode == 'he':
            W = np.random.randn(n_out, n_in) * np.sqrt(2.0 / n_in)
            
        Z = np.dot(W, A)
        
        if activation_fn == 'tanh':
            A = np.tanh(Z)
        elif activation_fn == 'relu':
            A = np.maximum(0, Z)
            
        variances.append(np.var(A))
        
    return variances

layers = np.arange(0, 11)
var_zero = simulate_propagation('zero', 'tanh')
var_unscaled = simulate_propagation('unscaled', 'tanh')
var_xavier = simulate_propagation('xavier', 'tanh')
var_he = simulate_propagation('he', 'relu')

plt.figure(figsize=(9, 5))
plt.plot(layers, var_unscaled, marker='o', color='#e03131', label='Unscaled Normal (Exploding)')
plt.plot(layers, var_zero, marker='x', color='#adb5bd', label='Zero Initialization')
plt.plot(layers, var_xavier, marker='s', color='#339af0', label='Xavier (Tanh)')
plt.plot(layers, var_he, marker='d', color='#2b8a3e', label='He (Kaiming ReLU)')
plt.yscale('log')
plt.title("Activation Variance Propagation across 10 Hidden Layers")
plt.xlabel("Layer Depth (l)")
plt.ylabel("Variance of Activations (Log Scale)")
plt.legend(frameon=True, facecolor='#ffffff')
plt.xticks(layers)
plt.grid(True, which="both", linestyle='--')
plt.show()

## 3. Empirical Training Stability: PyTorch BatchNorm Demo

Let's build a simple deep classification network and train it on synthetic data with vs without **Batch Normalization** to monitor convergence speeds.

In [ ]:
# Generate classification data
X_train = torch.randn(1000, 50)
y_train = torch.randint(0, 2, (1000,))

# Model WITHOUT BatchNorm
class NetWithoutBN(nn.Module):
    def __init__(self):
        super(NetWithoutBN, self).__init__()
        self.fc1 = nn.Linear(50, 100)
        self.fc2 = nn.Linear(100, 100)
        self.fc3 = nn.Linear(100, 100)
        self.fc4 = nn.Linear(100, 2)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

# Model WITH BatchNorm
class NetWithBN(nn.Module):
    def __init__(self):
        super(NetWithBN, self).__init__()
        self.fc1 = nn.Linear(50, 100)
        self.bn1 = nn.BatchNorm1d(100)
        self.fc2 = nn.Linear(100, 100)
        self.bn2 = nn.BatchNorm1d(100)
        self.fc3 = nn.Linear(100, 100)
        self.bn3 = nn.BatchNorm1d(100)
        self.fc4 = nn.Linear(100, 2)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.relu(self.bn3(self.fc3(x)))
        x = self.fc4(x)
        return x

def train_model(model, epochs=40):
    criterion = nn.CrossEntropyLoss()
    # Use SGD with a high learning rate to show stability differences
    optimizer = optim.SGD(model.parameters(), lr=0.1)
    loss_history = []
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())
        
    return loss_history

history_no_bn = train_model(NetWithoutBN())
history_bn = train_model(NetWithBN())

plt.figure(figsize=(7.5, 4.5))
plt.plot(history_no_bn, label='Without BatchNorm', color='#e03131', linewidth=2)
plt.plot(history_bn, label='With BatchNorm', color='#2b8a3e', linewidth=2.5)
plt.title("BatchNorm Stabilization Effect")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.legend(frameon=True, facecolor='#ffffff')
plt.grid(True)
plt.show()